## Example 1 — the installed version, and the recipe library that ships with it

In [ ]:
import importlib.metadata
import pathlib
import modelopt
import modelopt_recipes
from modelopt.recipe import load_recipe

assert modelopt.__version__ == modelopt_recipes.__version__ == importlib.metadata.version("nvidia_modelopt"), (
    modelopt.__version__, modelopt_recipes.__version__)
library = pathlib.Path(modelopt_recipes.__file__).parent
shipped = sorted(path.relative_to(library).as_posix() for path in library.rglob("*.yaml"))
assert "general/ptq/fp8_default-kv_fp8.yaml" in shipped and len(shipped) > 100, len(shipped)
recipe = load_recipe("general/ptq/fp8_default-kv_fp8")      # resolved inside the installed library, not from the network
assert type(recipe).__name__ == "ModelOptPTQRecipe", type(recipe)
print("nvidia_modelopt", modelopt.__version__, "with", len(shipped), "recipe files; loaded a", type(recipe).__name__)

## Example 2 — post-training quantization of a tiny model to 8-bit integers, on the CPU

In [ ]:
import torch
import modelopt.torch.quantization as mtq

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(16, 32), torch.nn.ReLU(), torch.nn.Linear(32, 4))
batches = [torch.randn(8, 16) for _ in range(4)]
reference = model(batches[0]).detach().clone()
def calibrate(candidate):
    for batch in batches:
        candidate(batch)
quantized = mtq.quantize(model, mtq.INT8_DEFAULT_CFG, forward_loop=calibrate)
quantizers = [name for name, module in quantized.named_modules() if type(module).__name__ == "TensorQuantizer"]
assert len(quantizers) == 6, quantizers                      # input, output and weight quantizer on each Linear
assert quantized[0].weight_quantizer.num_bits == 8 and quantized[0].weight_quantizer.amax is not None
error = float((quantized(batches[0]).detach() - reference).abs().max())
assert 0.0 < error < 0.1, error                              # changed by rounding, and only slightly
print("6 quantizers inserted and calibrated; largest output change after 8-bit quantization:", round(error, 4))

## Example 3 — the quantized state is saved and restored into a fresh model

In [ ]:
import io
import torch
import modelopt.torch.opt as mto
import modelopt.torch.quantization as mtq

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(16, 32), torch.nn.ReLU(), torch.nn.Linear(32, 4))
batches = [torch.randn(8, 16) for _ in range(4)]
quantized = mtq.quantize(model, mtq.INT8_DEFAULT_CFG, forward_loop=lambda candidate: [candidate(batch) for batch in batches])
expected = quantized(batches[0]).detach().clone()
saved = io.BytesIO()
mto.save(quantized, saved)
saved.seek(0)
fresh = torch.nn.Sequential(torch.nn.Linear(16, 32), torch.nn.ReLU(), torch.nn.Linear(32, 4))
assert not mto.ModeloptStateManager.is_converted(fresh)
restored = mto.restore(fresh, saved)
assert mto.ModeloptStateManager.is_converted(restored) and torch.equal(restored(batches[0]), expected)
print("saved", saved.getbuffer().nbytes, "bytes; the restored model gives identical outputs")

## Example 4 — magnitude sparsity: half of the first layer's weights are set to zero, the smaller half

In [ ]:
import copy
import torch
import modelopt.torch.sparsity as mts

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(16, 32), torch.nn.ReLU(), torch.nn.Linear(32, 4))
batches = [torch.randn(8, 16) for _ in range(4)]
dense = copy.deepcopy(model)
sparse = mts.sparsify(model, "sparse_magnitude")
weight, before = sparse[0].weight.detach(), dense[0].weight.detach()
dropped = weight == 0
assert abs(float(dropped.float().mean()) - 0.5) < 0.05, float(dropped.float().mean())
assert torch.equal(weight[~dropped], before[~dropped])                 # the kept weights are untouched
assert float(before[dropped].abs().mean()) < float(before[~dropped].abs().mean())   # the small ones were dropped
assert torch.isfinite(sparse(batches[0])).all()
print("first layer: zero fraction", round(float(dropped.float().mean()), 2), "; kept weights unchanged and larger on average")